<h1 style="color: #9f43c3ff; border-bottom: 3px solid #9f43c3ff; padding-bottom: 8px;">
AI & ML Course with BinX — Week 4 — Day 2: Cross-Validation & Stratification
</h1>

<blockquote style="border-left: 3px solid #2e1a9aff; padding-left: 12px; margin-left: 0;">

<b>Day 2 Learning Objectives:</b>
-  <b>$k$-Fold Cross-Validation:</b> Replace single validation splits with $k$ rotating folds for a stable, variance-aware performance estimate.
-  <b>Mean & Standard Deviation:</b> Interpret `cross_val_score` mean accuracy and standard deviation across folds.
-  <b>Stratified $k$-Fold:</b> Preserve multi-class target proportions across all training and validation folds.
- <b>Leak-Free Pipeline:</b> Ensure feature scaling via `StandardScaler` occurs strictly within each CV fold inside a `Pipeline`.
-  <b>Day 1 Comparison:</b> Benchmark the 5-fold CV estimate against Day 1's single validation split.

</blockquote>


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

## <span style="color: #309c42ff">2.1 Dataset Loading & Inspection</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

We load the pre-cleaned (from week 3) Palmer Archipelago Penguins dataset (<code>penguins_cleaned.csv</code>) containing 333 complete records across 3 species (Adelie, Gentoo, Chinstrap).

</blockquote>


In [2]:
df = pd.read_csv("../../Data/penguins_cleaned.csv")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.info()
print("\nUnique values in 'sex':", df['sex'].unique())

Dataset Shape: 333 rows, 7 columns

<class 'pandas.DataFrame'>
RangeIndex: 333 entries, 0 to 332
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            333 non-null    str    
 1   island             333 non-null    str    
 2   culmen_length_mm   333 non-null    float64
 3   culmen_depth_mm    333 non-null    float64
 4   flipper_length_mm  333 non-null    float64
 5   body_mass_g        333 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.3 KB

Unique values in 'sex': <StringArray>
['MALE', 'FEMALE']
Length: 2, dtype: str


## <span style="color: #309c42ff">2.2 Feature Encoding & 80/20 Partitioning</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

We apply one-hot dummy encoding to categorical columns (`island`, `sex`) and hold out <b>20% as an unseen test set</b>. The remaining <b>80% (`X_train_cv`, `y_train_cv`)</b> will be used exclusively for 5-Fold Cross-Validation.

</blockquote>


In [3]:
df_encoded = pd.get_dummies(df, columns=['island', 'sex'], drop_first=True)

y = df_encoded['species']
X = df_encoded.drop(columns=['species'])

X_train_cv, X_test, y_train_cv, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## <span style="color: #309c42ff">2.3 Leak-Free Pipeline & 5-Fold Cross-Validation</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>Hands-On Lab — Step 1 & 2:</b>
We build a Scikit-Learn <code>Pipeline</code> containing <code>StandardScaler</code> and <code>KNeighborsClassifier(n_neighbors=1)</code>. The pipeline guarantees that scaling parameters ($\mu, \sigma$) are computed strictly from the 4 training folds in each iteration, completely eliminating validation fold leakage.

</blockquote>


In [4]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=1))
])

scores = cross_val_score(pipeline, X_train_cv, y_train_cv, cv=5, scoring="accuracy")


In [ ]:
print("--- 5-Fold Cross-Validation Scores ---")
for fold, score in enumerate(scores, 1):
    print(f"Fold {fold}: {score:.4f}")

print("-" * 35)
print(f"Mean CV Accuracy:   {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")


--- 5-Fold Cross-Validation Scores ---
Fold 1: 1.0000
Fold 2: 0.9811
Fold 3: 1.0000
Fold 4: 0.9811
Fold 5: 1.0000
-----------------------------------
Mean CV Accuracy:   0.9925
Standard Deviation: 0.0092


## <span style="color: #309c42ff">2.4 Comparison with Day 1 Single Validation Split</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>Hands-On Lab — Step 3:</b>
We compare the single-split validation score from Day 1 ($98.51\%$) with our 5-fold cross-validated mean and standard deviation ($0.9925 \pm 0.0092$).

</blockquote>


In [6]:
single_split_score = 0.9851  # Day 1 validation accuracy (1 split)

print("--- Comparison with Day 1 ---")
print(f"Day 1 Single-Split Score: {single_split_score:.4f}")
print(f"Day 2 5-Fold CV Score:    {scores.mean():.4f} ± {scores.std():.4f}")

--- Comparison with Day 1 ---
Day 1 Single-Split Score: 0.9851
Day 2 5-Fold CV Score:    0.9925 ± 0.0092


## <span style="color: #309c42ff">2.5 Verification of Stratified $k$-Fold Distribution</span>

<blockquote style="border-left: 3px solid #FD1D1D; padding-left: 12px; margin-left: 0;">

<b>Hands-On Lab — Step 4:</b>
We inspect the target class distribution across each fold to confirm that <code>StratifiedKFold</code> maintains identical species proportions across all 5 validation splits.

</blockquote>


In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Step 4: Class Proportions per Fold (%) ---")
for folds, (_, val_idx) in enumerate(skf.split(X_train_cv, y_train_cv), 1):
    proportions = y_train_cv.iloc[val_idx].value_counts(normalize=True) * 100
    print(f"Fold {folds}: " + ", ".join([f"{cls}: {pct:.1f}%" for cls, pct in proportions.items()]))


--- Step 4: Class Proportions per Fold (%) ---
Fold 1: Adelie: 42.6%, Gentoo: 37.0%, Chinstrap: 20.4%
Fold 2: Adelie: 43.4%, Gentoo: 35.8%, Chinstrap: 20.8%
Fold 3: Adelie: 43.4%, Gentoo: 35.8%, Chinstrap: 20.8%
Fold 4: Adelie: 43.4%, Gentoo: 35.8%, Chinstrap: 20.8%
Fold 5: Adelie: 43.4%, Gentoo: 35.8%, Chinstrap: 20.8%


## <span style="color: #9f43c3ff">2.6 Day 2 Lab Reflection: Cross-Validation vs. Single Split</span>

<blockquote style="border-left: 3px solid #2e1a9aff; padding-left: 12px; margin-left: 0;">

### 1. What Cross-Validation Does
A single validation set has a fundamental weakness: if it happens to be an unusual slice of the data, tuning decisions will be based on luck rather than signal. $k$-fold cross-validation splits the training data into $k$ equal parts (folds), training $k$ times such that each fold serves as the validation set once while the other $k-1$ folds are used for training.

### 2. Interpreting the Mean and Standard Deviation
- **The Mean ($0.9925$):** Serves as the primary performance estimate across all rotating subsets.
- **The Standard Deviation ($\pm 0.0092$):** Measures performance stability across folds. A high mean with a low standard deviation indicates a model you can trust.

### 3. Why Stratified Folds Matter
In the Palmer Penguins dataset, the target classes are distributed as ~43.8% Adelie, ~35.7% Gentoo, and ~20.4% Chinstrap. Stratified $k$-fold ensures that every fold maintains these proportions, preventing folds from having disproportionately few minority class samples.

### 4. Leakage Prevention via Scikit-Learn Pipelines
If scaling is applied before splitting or across folds, validation information leaks into training. Chaining `StandardScaler` and `KNeighborsClassifier` inside a `Pipeline` ensures each fold is scaled using only that fold's training portion, making data leakage structurally impossible.

</blockquote>
